# 第 22 课：WFST 组合——L、G、经典 HCLG 与 CTC 解码图

目标：知道每张图负责什么，并避免把经典 HMM 解码图原样套到 CTC。

<!-- course-upgrade-v2 -->
## 学习导航

| 项目 | 内容 |
|---|---|
| 所属阶段 | 语言模型与 WFST |
| 建议投入 | 3～5 小时，可分 2～3 次完成 |
| 前置要求 | 完成第 21 课；如果前测低于 2/3，先回看上一课小结 |
| 本课核心 | L 图、G 图、HCLG 与 CTC TLG |
| 完成标准 | 能口头解释核心概念；独立完成强化题；从空白重写核心函数 |

高效顺序：**先回答前测 → 预测代码结果 → 再运行 → 修改一个变量 → 关闭答案复现 → 次日回忆。**


<!-- course-upgrade-v2 -->
## 课前诊断（先不要运行代码）

1. 分别用一句话解释：L 图、G 图、HCLG 与 CTC TLG。
2. 画出这三个概念之间的输入—输出关系。
3. 写下你最不确定的一点，并给出一个暂时猜测。

自评：答对 0～1 题先复习前置课；答对 2 题可以正常学习；3 题都能讲清楚则直接挑战代码和迁移题。


<!-- course-bridge-v3 -->
## 知识接力：先取回旧知识，再进入本课

### 3 分钟闭卷回忆

在新 Markdown cell 中回答，**不要先翻前文**：CTC beam 中的候选前缀；概率与负对数代价方向；开发集和测试集权限。

- 三项都能用“含义 + 单位/shape + 一个数字例子”回答：进入本课。
- 能回答两项：学习本课，但把缺口记入 `LEARNING_LOG.md`。
- 只能回答零到一项：先回到 [上一课](21_FSA与WFST_状态弧权重和最短路径.ipynb)与[唯一学习路径](../LEARNING_PATH.md)，做一次最小实验；不要靠继续看新术语掩盖断点。

### 本课接口契约

```text
输入：声学候选、语言模型/图状态、分数权重
  ↓ 本课要学会的变换、状态或判断
输出：可解释、可冻结调参、可分块保持状态的上下文解码结果
```

学完后必须能解释：输入的哪个单位/shape/状态若丢失，会让输出“仍能运行却语义错误”。


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

def find_root():
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "pyproject.toml").exists(): return p
    raise FileNotFoundError("请从 learn_asr 或 notebooks 目录启动 Jupyter")

ROOT = find_root()
BLANK = "∅"
plt.rcParams["figure.figsize"] = (11, 4)
print("项目根目录:", ROOT)

## 1. L：词典 transducer

`L` 把音素或模型 token 序列映射成词。例如：

```text
jin tian  → 今天
qi        → 气
tian qi   → 天气
```

一个词可能有多个发音，一个发音也可能产生歧义，因此它天然适合用 transducer 表示。

## 2. G：语言模型 acceptor

`G` 限制并给词序列打分，例如 `今天天气` 比 `今天田气` 代价低。N-gram backoff 通常也编码成特殊的 epsilon/回退弧。

In [ ]:
words=["今天","天气","很好"]
G={("<s>","今天"):.1,("今天","天气"):.2,("天气","很好"):.15,
   ("<s>","天气"):1.5,("天气","今天"):2.0}
def graph_cost(seq):
    prev="<s>";cost=0
    for w in seq: cost+=G.get((prev,w),3.0);prev=w
    return cost
for seq in [["今天","天气","很好"],["天气","今天"]]:print(seq,graph_cost(seq))

## 3. 经典 Kaldi HCLG

```text
H：HMM 拓扑/transition-id → 上下文相关音素
C：上下文相关音素 → 普通音素
L：音素 → 词
G：词语言模型

H ∘ C ∘ L ∘ G
```

这是传统 HMM-DNN 系统的结构。`G` 是语言模型，`L` 是发音词典。

## 4. CTC 解码图不一定叫 HCLG

端到端 CTC 已经没有传统 HMM state 拓扑和 context-dependent phone 的同一含义。常见概念图可能是：

```text
T：CTC blank/repeat topology
L：token/phone → word lexicon（字级系统可非常简单）
G：word/character language model

T ∘ L ∘ G
```

不同工具包命名不同。核心问题不是背缩写，而是逐张确认输入符号、输出符号和权重。

## 5. Composition 的接口思维

只有当前一张图的 output label 能与后一张图的 input label 匹配，两张图才能组合。任何 WFST 排错都先检查 symbol table 和标签空间。

In [ ]:
layers=[("CTC posterior","token id"),("T","normalized token"),("L","word"),("G","word score")]
fig,ax=plt.subplots(figsize=(11,2.5))
for i,(name,out) in enumerate(layers):
    ax.scatter(i,0,s=1800,color=f"C{i}");ax.text(i,0,name,ha="center",va="center")
    if i<len(layers)-1: ax.annotate(out,(i+1-.18,0),(i+.18,0),arrowprops=dict(arrowstyle="->"),ha="center")
ax.set(xlim=(-.5,len(layers)-.5),ylim=(-.7,.7),title="Conceptual CTC decoding pipeline");ax.axis("off");plt.show()

## 本课测试

1. `L` 和 `G` 分别负责什么？
2. HCLG 中的 H/C 是否能不加解释地套到 CTC？
3. 字级 CTC 是否一定需要复杂发音词典？
4. composition 失败首先检查什么？
5. 为什么要把大图 determinize/minimize？

<details><summary>展开参考答案</summary>

1. L 做发音/token 到词映射，G 对词/token 序列建模。2. 不能，模型拓扑不同。3. 不一定。4. 相邻图的输入/输出标签空间和 symbol table。5. 减小图和搜索状态，提高解码速度。

</details>

<!-- course-upgrade-v2 -->
## 强化练习：第 22 课专属题库

请先把答案写进新的 Markdown/Code cell，再展开自评标准。

### A. 基础回忆

1. 不看上文，分别定义 `L 图`、`G 图`、`HCLG 与 CTC TLG`。
2. 哪一个量/状态是本课最容易在模块边界丢失的？它的单位和 shape 是什么？
3. 本课至少写出两个“看起来能运行，但结果其实错误”的例子。

### B. 预测与推理

4. 场景：**相邻 FST symbol table 不一致**。先预测现象，再说明原因，最后给出一项可以验证猜测的指标。
5. 改变本课最关键参数的 0.5×、1×、2×，分别预测准确率、延迟、内存或数值误差怎样变化。
6. 画一张最小数据流图，在每条边标出 dtype、shape、时间单位或概率/代价方向。

### C. 编程与排错

7. 编程任务：**画出每张图输入/输出标签并检查 composition**。至少加入正常、边界、错误输入三类测试。
8. 故意制造一个 off-by-one、shape、状态未 reset 或数值稳定性错误；记录错误现象和定位过程。
9. 不看本课实现，从空白 cell 重写最核心函数，并用原实现作数值对照。

### D. 迁移与表达

10. 跨课任务：**解释经典 HCLG 哪些部分不能直接套 CTC**。
11. 用 90 秒向没有学过 ASR 的人解释本课；禁止只念术语，必须举一个数字或生活例子。
12. 写出一个生产系统中会监控的指标，以及它异常时优先检查的三处位置。

<details><summary>展开自评标准</summary>

- 每题 0～2 分：0=无法回答；1=方向正确但缺少单位、边界或验证；2=解释完整且能用代码/数字验证。
- 24 分满分：达到 19 分再进入下一课；15～18 分次日重做错题；低于 15 分回看本课图和核心代码。
- 第 4 题必须包含“预测—原因—指标”，第 7～9 题必须真正运行测试，第 10 题必须明确上下游 contract。
- 核心答案至少应正确使用：L 图、G 图、HCLG 与 CTC TLG。

</details>


<!-- course-upgrade-v2 -->
## 间隔复习与离场票

### 离场票（现在完成）

- [ ] 我能不用笔记解释 L 图、G 图、HCLG 与 CTC TLG。
- [ ] 我能说出本课最常见的错误及其观测现象。
- [ ] 我能从空白重写一个核心函数，并通过至少 3 个测试。
- [ ] 我能说明本课对上一层和下一层接口的影响。

### 复习时间表

- **明天（5 分钟）**：闭卷写出三个核心概念和一个公式/shape。
- **7 天后（15 分钟）**：重做第 4、7、10 题，不运行原答案。
- **30 天后（20 分钟）**：从真实音频或随机张量重新构造一个最小实验。

把错题记录到根目录 `LEARNING_LOG.md`。不要只写“不会”，要写：原判断、证据、正确规则、下次检查动作。
